# 02 — Dados externos (indústria de laticínios, bioetanol, economia)

## Objetivo

O notebook `01_data_audit` mostrou que o experimento sozinho não permite:
- estimar quanto soro uma escala industrial geraria (não é um dado de laticínio);
- calcular um teto teórico de conversão lactose→etanol (precisa de estequiometria + eficiência de literatura);
- estimar custos e receita industriais (precisa de preços de mercado).

Este notebook documenta **cada dado externo usado**, com fonte, URL, data, região, metodologia e nível de confiabilidade — e também **os dados que foram procurados e não encontrados** com qualidade suficiente, para que isso não vire uma suposição escondida mais adiante.

A tabela completa está em `data/external/dados_externos.csv`; a metodologia de busca priorizou artigos revisados por pares, instituições de pesquisa (EPAMIG, CNPEM/CTBE) e fontes de mercado reconhecidas (CEPEA/Esalq), evitando blogs sem metodologia declarada.


In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import models as m

pd.set_option("display.max_colwidth", 80)

ext = m.load_external_data()
print(f"{len(ext)} dados externos catalogados, em {ext['categoria'].nunique()} categorias.")
ext["categoria"].value_counts()

18 dados externos catalogados, em 3 categorias.


categoria
economia      7
laticinios    6
bioetanol     5
Name: count, dtype: int64

## Indústria de laticínios (soro de leite)


In [2]:
cols_show = ["id", "variavel", "valor", "unidade", "fonte", "confiabilidade"]
ext[ext["categoria"] == "laticinios"][cols_show]

,id,variavel,valor,unidade,fonte,confiabilidade
0,E01,proporcao_soro_por_kg_queijo,7-9,L soro / kg queijo,"Paula, J.C.J. et al. Adequabilidade de diferentes tipos de soros de leite pa...",Alta (instituicao de pesquisa EPAMIG/FAPEMIG; fonte primaria citada Walstra ...
1,E02,soro_percentual_volume_leite,85-90,% do volume de leite usado no queijo,"Paula, J.C.J. et al. EPAMIG-ILCT (citando Gonzalez Siso 1996; Atra et al. 2005)",Alta
2,E03,composicao_soro_lactose,4.9,% m/m (~49 g/L),"Paula, J.C.J. et al. EPAMIG-ILCT, Tabela 2 (adaptado de Mello 1989; Fox 1997...",Alta
3,E04,composicao_soro_umidade,93.4,% m/m,"Paula, J.C.J. et al. EPAMIG-ILCT, Tabela 2",Alta
4,E05,DBO_soro_faixa,25000-120000,mg/L,"Paula, J.C.J. et al. EPAMIG-ILCT (citando Pelegrine e Carrasqueira 2008; Gir...",Alta
5,E06,DBO_soro_valor_especifico,45000,mg/L,MilkPoint/EPAMIG - coluna Soro do queijo: residuo ou materia-prima? (citando...,"Media (fonte secundaria - coluna de opiniao tecnica, mas assinada por especi..."


**Achado central**: o artigo de Paula et al. (EPAMIG — Instituto de Laticínios Cândido Tostes, financiado pela FAPEMIG) informa que **o soro representa 85–90% do volume de leite usado no queijo** e que **cada kg de queijo gera de 7 a 9 L de soro** (citando Walstra, 2006). Isso **corrobora de forma independente** a faixa de 80–90% já citada no próprio relatório do projeto — um bom sinal de que o número é consolidado na literatura, não um caso isolado.

A composição média do soro (mesma fonte, Tabela 2, adaptada de Mello 1989; Fox 1997; Walstra et al. 2006) dá **lactose ≈ 4,9% m/m (~49 g/L)**. Esse valor **não é medição do soro usado neste projeto** — é um proxy de literatura, necessário porque a concentração de lactose do nosso soro nunca foi medida (notebook 01).


## Processo de bioetanol (rendimentos, eficiências)


In [3]:
ext[ext["categoria"] == "bioetanol"][cols_show]

,id,variavel,valor,unidade,fonte,confiabilidade
6,E07,rendimento_teorico_estequiometrico,0.538,g etanol / g lactose,Calculo estequiometrico proprio (Gay-Lussac estendido: lactose + H2O -> 2 he...,Alta (estequiometria pura; nao e dado experimental externo)
7,E08,eficiencia_fermentacao_industrial_min,80,% do rendimento teorico,Sintese de multiplos estudos sobre fermentacao de soro de leite/lactose por ...,Media (dados obtidos via resumos/abstracts indexados; texto completo nao ace...
8,E09,eficiencia_fermentacao_industrial_max,97,% do rendimento teorico,Mesmos estudos de E08 (maior valor reportado: 97% do rendimento teorico em 13h),Media (mesma ressalva de E08)
9,E10,concentracao_etanol_literatura,11.7-24.85,g/L,Estudos de fermentacao de soro de queijo (crude/sweet whey) por Kluyveromyce...,"Media (via resumo indexado, texto completo nao acessado)"
10,E11,artigo_soro_ricota_etanol_referencia,NaN,NaN,"Sansonetti, S.; Curcio, S.; Calabro, V.; Iorio, G. Bio-ethanol production by...","Nao verificavel (paywall - so o registro bibliografico foi confirmado, nao o..."


**Rendimento teórico estequiométrico (E07)** — este não é uma citação de terceiros, é um **cálculo nosso**, direto da equação química:

$$ \text{C}_{12}\text{H}_{22}\text{O}_{11} + \text{H}_2\text{O} \xrightarrow{\text{lactase}} 2\ \text{hexoses} \xrightarrow{\text{fermentação}} 4\ \text{C}_2\text{H}_5\text{OH} + 4\ \text{CO}_2 $$

$$ \text{rendimento teórico} = \frac{4 \times M_{\text{etanol}}}{M_{\text{lactose}}} = \frac{4 \times 46{,}07}{342{,}3} \approx 0{,}538\ \text{g etanol / g lactose} $$

Esse número é o **teto absoluto** de qualquer processo de fermentação de lactose — nenhuma rota real chega a 100% dele.

**Eficiência industrial de fermentação (E08–E10)**: estudos publicados relatam 80–97% do rendimento teórico, com concentrações de 11,7 a 24,85 g/L de etanol no mosto. **Ressalva importante**: esses estudos usam a levedura *Kluyveromyces marxianus/lactis*, que fermenta a lactose **diretamente**, sem hidrólise prévia — uma rota biológica **diferente** da usada neste projeto (hidrólise com lactase + *Saccharomyces cerevisiae*, que não metaboliza lactose). Servem como **teto de comparação tecnológica**, não como validação direta da nossa rota.

**Sansonetti et al. (2009) (E11)** é a referência bibliográfica mais próxima do nosso processo (etanol a partir de soro de **ricota**, ou seja, soro já desproteinado — igual ao nosso filtrado). O artigo está com o texto completo bloqueado (paywall); registramos apenas a referência, **sem usar nenhum valor numérico dele**, porque não pudemos verificá-lo.


## Economia (preços de mercado, energia, insumos)


In [4]:
ext[ext["categoria"] == "economia"][cols_show]

,id,variavel,valor,unidade,fonte,confiabilidade
11,E12,preco_etanol_hidratado_CEPEA,2.6412,BRL/L,Indicador CEPEA/Esalq - Etanol Hidratado Combustivel (usinas de Sao Paulo),Alta (CEPEA-Esalq/USP e referencia de mercado reconhecida no setor sucroener...
12,E13,preco_etanol_anidro_CEPEA,2.9128,BRL/L,Indicador CEPEA/Esalq - Etanol Anidro,Alta
13,E14,custo_producao_etanol_1G,1.15,BRL/L,"Agencia FAPESP, citando Antonio Bonomi (CTBE/CNPEM)",Alta (instituicao de pesquisa federal CNPEM/CTBE)
14,E15,custo_producao_etanol_2G_enzimas,20-40,% do custo total de producao,"Agencia FAPESP, citando CTBE/CNPEM",Alta
15,E16,tarifa_energia_industrial_CEMIG_A4,consumo ponta 0.66001 / fora-ponta 0.48077 (BRL/kWh); demanda 23.77 (BRL/kW),BRL/kWh e BRL/kW,"CEMIG - Tarifas Vigentes (Grupo A4, bandeira verde)",Alta (fonte oficial da concessionaria que atende a regiao do projeto)
16,E17,preco_lactase_varejo,40.00,BRL / caixa de 30 capsulas de 376 mg,Dado do proprio relatorio do projeto (compra em farmacia),Alta (dado primario do proprio projeto)
17,E18,preco_fermento_varejo,2.50,BRL / pacote (comercio local),Dado do proprio relatorio do projeto,Alta (dado primario do proprio projeto)


**Preço do etanol (E12/E13)**: CEPEA/Esalq — referência de mercado usada pela cadeia sucroenergética brasileira. Hidratado R$ 2,6412/L na semana de 14–18/09/2026 (variação semanal de +7,3%, o que já indica a volatilidade do indicador — relevante para a análise de sensibilidade do notebook 05).

**Tarifa de energia (E16)**: CEMIG — mesma concessionária da região de Lavras/MG onde o projeto foi executado. Tarifa binomial do Grupo A4 (consumo + demanda), bandeira verde.

**Custo de enzimas em bioetanol 2G (E14/E15)**: dados do CTBE/CNPEM (instituição pública de pesquisa), mas para **etanol de segunda geração a partir de bagaço de cana** — processo diferente do nosso (celulose vs. lactose). Usamos apenas como referência de ordem de grandeza (enzimas tipicamente representam 20–40% do custo total em um processo enzimático de produção de etanol), não como valor direto.

### O que NÃO conseguimos encontrar (lacunas declaradas)

- **Preço industrial de lactase a granel** (R$/kg ou R$/U.FCC): não encontramos fornecedor ou publicação com preço industrial. Vamos usar o preço de **varejo/farmácia já pago pelo próprio projeto** (E17: R$ 40,00 pela caixa de 30 cápsulas) como proxy — sabendo que isso tende a **superestimar** fortemente o custo real em escala industrial, já que preço de varejo farmacêutico costuma ser ordens de grandeza maior que insumo a granel.
- **Preço industrial de levedura seca a granel**: mesma situação (E18, R$ 2,50/pacote de comércio local).

Essas duas lacunas serão o alvo principal da análise de sensibilidade (notebook 05), porque são, ao mesmo tempo, os parâmetros mais incertos e — como veremos — os que mais pesam no resultado econômico.


## Resumo — o que este notebook entrega para os próximos

| Parâmetro | Valor adotado | Origem | Confiabilidade |
|---|---|---|---|
| Proporção soro:queijo | 7–9 L/kg | EPAMIG/ILCT (Walstra 2006) | Alta |
| Lactose no soro | ~49 g/L | EPAMIG/ILCT (Mello/Fox/Walstra) | Alta (mas é proxy, não medição do nosso soro) |
| Rendimento teórico lactose→etanol | 0,538 g/g | Cálculo próprio (estequiometria) | Alta |
| Eficiência industrial de fermentação (referência) | 80–97% do teórico | Literatura (*Kluyveromyces*) | Média (rota biológica diferente) |
| Preço do etanol | R$ 2,64/L | CEPEA/Esalq, set/2026 | Alta (mas volátil) |
| Tarifa de energia industrial | CEMIG A4 | CEMIG, set/2026 | Alta |
| Preço de lactase e fermento | Varejo (R$ 40 / R$ 5) | Próprio relatório do projeto | Baixa como proxy industrial — declarado |

Seguimos para `03_technical_model`, onde esses dados entram na função `calculate_ethanol_production` de `src/models.py`.
